In [10]:
import pandas as pd
import numpy as np
import openpyxl
import re


In [11]:
#Reading excel file

file_name = "Example_1.xls"

df_reinf = pd.read_excel(file_name, sheet_name= "Summary")#, skiprows= [0, 2] )
#Removing the Layer values different than ARM2
df_reinf = df_reinf[df_reinf["Layer"] == "ARM2" ]


WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero


In [12]:
#Removing column "Layer"
df_reinf.drop("Layer",axis=1, inplace = True)


In [13]:
#Split the value column to list of values Count/Diameter/Length
def split_value(a):
    result = re.split(r'[CNx]', a)
    return result
df_reinf["List"] = df_reinf["Value"].apply(split_value)
df_reinf = df_reinf.reset_index(drop=True)
df_reinf.head(20)

,Count,Name,Value,List
0,1,Text,12N12x370,"[12, 12, 370]"
1,1,Text,24N12x370,"[24, 12, 370]"
2,1,Text,11N12x600,"[11, 12, 600]"
3,2,Text,8N12x350,"[8, 12, 350]"
4,1,Text,6N12x330,"[6, 12, 330]"
5,1,Text,9N14x450,"[9, 14, 450]"
6,4,Text,8N12x400,"[8, 12, 400]"
7,6,Text,8N12x580,"[8, 12, 580]"
8,1,Text,14N12x460,"[14, 12, 460]"
9,1,Text,22N14x460,"[22, 14, 460]"


In [14]:
def write_value(list, position):
    if len(list) == 4:
        value = list[position].strip("%")
        value = float(value)
    else:
        if position == -4:
            value = 1
        else:
            value = list[position].strip("%")
            value = float(value)
    return value
df_reinf["Multiplier"] = df_reinf["List"].apply(write_value, args = (-4,))
df_reinf["Number of pieces"] = df_reinf["List"].apply(write_value, args = (-3,))
df_reinf["Diameter"] = df_reinf["List"].apply(write_value, args = (-2,))
df_reinf["Length"] = df_reinf["List"].apply(write_value, args = (-1,))
#df_reinf.head(20)

In [15]:
df_reinf.drop(["Name", "List"], axis=1, inplace = True)
#df_reinf.head(20)

In [16]:
#Calculating the weigth 
df_reinf["Weight"] = df_reinf.apply(lambda row: (3.1415926*(row["Diameter"]/2)*(row["Diameter"]/2))
                                    *row["Length"]*10
                                    *row["Number of pieces"]
                                    *row["Multiplier"]
                                    *row["Count"], axis=1)
df_reinf["Weight"] = df_reinf["Weight"].apply(lambda x: x*0.00000785)
# Calculating the Total Length
df_reinf["TotalLength"] = df_reinf.apply(lambda row: row["Length"]
                                    *row["Number of pieces"]
                                    *row["Multiplier"]
                                    *row["Count"], axis=1)
#df_reinf.head(20)

In [17]:
#Creating data frame with each diameter weight
diameters = [6.5, 8, 10, 12, 14, 16, 18, 20, 22, 25, 28, 30, 32, 34, 36, 38]
weight_reinf = {}
for d in diameters:
    weight = df_reinf[df_reinf["Diameter"]== d]["Weight"].sum()
    weight_reinf [f"N{d}"] = list()
    weight_reinf [f"N{d}"].append("B500")
    weight_reinf [f"N{d}"].append(df_reinf[df_reinf["Diameter"]== d]["TotalLength"].sum())
    weight_reinf [f"N{d}"].append(round(weight, 1))
df_weight_reinf = pd.DataFrame.from_dict(weight_reinf, orient="index", columns = ["Class", "Length", "Weight"])
#df_weight_reinf.head(7)


In [18]:
export_file_name = 'Retaining_walls_OUT_Check.xlsx'

df_weight_reinf.to_excel(export_file_name, index=True)

In [13]:
#Creating data frame for each diameter - Used in case I want to check or the contractor wants it. 
#diameters = [8, 10, 12, 14, 16, 18, 20, 22, 25, 28, 30, 32, 34, 36, 38]
#diameter_reinf = {}
#for d in diameters:
    #diameter_reinf[f"N{d}"] = df_reinf[df_reinf["Diameter"]== d].reset_index(drop=True)

#diameter_reinf["N8"].head(4)

,Count,Value,Multiplier,Number of pieces,Diameter,Length,Weight,TotalLength
0,1,2N8x112,1,2.0,8.0,112.0,0.883868,224.0
1,1,22N8x95,1,22.0,8.0,95.0,8.246806,2090.0


In [59]:
#diameter_reinf["N14"].to_excel('N14.xlsx', index=False)